In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

np.random.seed(42)
tf.random.set_seed(42)


def get_user_input():
    image_path = input("Enter path to the image file: ").strip()

    while not os.path.isfile(image_path):
        image_path = input("File not found. Enter a valid path: ").strip()

    return image_path


def load_and_preprocess(image_path, img_size):
    original_file_size = os.path.getsize(image_path)

    img = Image.open(image_path).convert("RGB")

    original_dims = img.size

    img = img.resize((img_size, img_size))

    img_array = np.array(img).astype("float32") / 255.0

    return img_array, original_file_size, original_dims


def add_noise(image, noise_factor):
    noise = np.random.normal(
        loc=0.0,
        scale=noise_factor,
        size=image.shape
    )

    noisy_image = image + noise

    noisy_image = np.clip(
        noisy_image,
        0.0,
        1.0
    )

    return noisy_image


def build_autoencoder(img_size):

    inputs = layers.Input(
        shape=(img_size, img_size, 3)
    )

    x = layers.Conv2D(
        32,
        (3, 3),
        activation="relu",
        padding="same"
    )(inputs)

    x = layers.MaxPooling2D(
        (2, 2),
        padding="same"
    )(x)

    x = layers.Conv2D(
        64,
        (3, 3),
        activation="relu",
        padding="same"
    )(x)

    encoded = layers.MaxPooling2D(
        (2, 2),
        padding="same"
    )(x)

    encoder = models.Model(
        inputs,
        encoded
    )

    encoded_inputs = layers.Input(
        shape=encoded.shape[1:]
    )

    x = layers.Conv2D(
        64,
        (3, 3),
        activation="relu",
        padding="same"
    )(encoded_inputs)

    x = layers.UpSampling2D(
        (2, 2)
    )(x)

    x = layers.Conv2D(
        32,
        (3, 3),
        activation="relu",
        padding="same"
    )(x)

    x = layers.UpSampling2D(
        (2, 2)
    )(x)

    decoded = layers.Conv2D(
        3,
        (3, 3),
        activation="sigmoid",
        padding="same"
    )(x)

    decoder = models.Model(
        encoded_inputs,
        decoded
    )

    outputs = decoder(encoder(inputs))

    autoencoder = models.Model(
        inputs,
        outputs
    )

    autoencoder.compile(
        optimizer=optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mse"
    )

    return autoencoder, encoder, decoder


class FirstLastEpochLogger(tf.keras.callbacks.Callback):

    def __init__(self, autoencoder, noisy_data, total_epochs):
        super().__init__()

        self.autoencoder_model = autoencoder
        self.noisy_data = noisy_data
        self.total_epochs = total_epochs

        self.first_output = None
        self.last_output = None

        self.first_loss = None
        self.last_loss = None


    def on_epoch_end(self, epoch, logs=None):

        output = self.autoencoder_model.predict(
            self.noisy_data,
            verbose=0
        )

        if epoch == 0:

            self.first_output = output[0].copy()

            self.first_loss = logs["loss"]


        if epoch == self.total_epochs - 1:

            self.last_output = output[0].copy()

            self.last_loss = logs["loss"]


def compute_mse(original, reconstructed):

    return np.mean(
        (original - reconstructed) ** 2
    )


def compute_rmse(mse):

    return np.sqrt(mse)


def compute_psnr(mse):

    if mse == 0:
        return float("inf")

    return 10 * np.log10(
        1.0 / mse
    )


def format_values(values, count=20):

    flat_values = values.flatten()

    selected = flat_values[:count]

    return " ".join(
        f"{value:.6f}"
        for value in selected
    )


def main():

    img_size = 128

    noise_factor = 0.25

    epochs = 50

    batch_size = 10

    training_samples = 100


    image_path = get_user_input()


    original_image, original_file_size, original_dims = load_and_preprocess(
        image_path,
        img_size
    )


    print("\nIMAGE INFORMATION")

    print(
        "Original Resolution:",
        original_dims[0],
        "x",
        original_dims[1]
    )

    print(
        "Resized Image:",
        img_size,
        "x",
        img_size,
        "x 3"
    )

    print(
        "Total Pixel Values:",
        img_size * img_size * 3
    )

    print(
        "Original File Size:",
        round(original_file_size / 1024, 2),
        "KB"
    )


    original_input = np.expand_dims(
        original_image,
        axis=0
    )


    noisy_input = add_noise(
        original_input,
        noise_factor
    )


    print("\nNOISE INFORMATION")

    print(
        "Noise Type: Gaussian Noise"
    )

    print(
        "Noise Factor:",
        noise_factor
    )


    autoencoder, encoder, decoder = build_autoencoder(
        img_size
    )


    print("\nAUTOENCODER ARCHITECTURE")

    print(
        "Input:",
        img_size,
        "x",
        img_size,
        "x 3"
    )

    print(
        "Encoder: Conv32 -> Pool -> Conv64 -> Pool"
    )

    print(
        "Encoded Representation:",
        img_size // 4,
        "x",
        img_size // 4,
        "x 64"
    )

    print(
        "Decoder: Conv64 -> Upsample -> Conv32 -> Upsample -> RGB"
    )


    x_train = np.repeat(
        original_input,
        training_samples,
        axis=0
    )


    noise_train = np.random.normal(
        0,
        noise_factor,
        x_train.shape
    )


    x_train_noisy = np.clip(
        x_train + noise_train,
        0,
        1
    )


    logger = FirstLastEpochLogger(
        autoencoder,
        noisy_input,
        epochs
    )


    print("\nTRAINING DENOISING AUTOENCODER")


    history = autoencoder.fit(

        x_train_noisy,

        x_train,

        epochs=epochs,

        batch_size=batch_size,

        shuffle=True,

        verbose=0,

        callbacks=[logger]

    )


    print("\nDENOISING TRAINING LOSS")

    print(
        "First Epoch Loss:",
        "{:.8f}".format(
            logger.first_loss
        )
    )

    print(
        "Last Epoch Loss:",
        "{:.8f}".format(
            logger.last_loss
        )
    )


    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        range(1, epochs + 1),
        history.history["loss"],
        linewidth=2
    )

    plt.xlabel("Epoch")

    plt.ylabel("MSE Loss")

    plt.title(
        "Denoising Autoencoder Training Loss"
    )

    plt.grid(True)

    plt.tight_layout()

    plt.show()


    print("\nDENOISING VALUES")


    print("\nEpoch 1 Output Values:")

    print(
        format_values(
            logger.first_output
        )
    )


    print("\nEpoch 50 Output Values:")

    print(
        format_values(
            logger.last_output
        )
    )


    print("\nENCODING IMAGE")


    encoded_image = encoder.predict(
        noisy_input,
        verbose=0
    )


    print(
        "Encoded Shape:",
        encoded_image.shape
    )


    print("\nDECODING AND DENOISING IMAGE")


    denoised_image = decoder.predict(
        encoded_image,
        verbose=0
    )


    denoised_image = denoised_image[0]


    noisy_image = noisy_input[0]


    mse_noisy = compute_mse(
        original_image,
        noisy_image
    )


    mse_denoised = compute_mse(
        original_image,
        denoised_image
    )


    rmse_noisy = compute_rmse(
        mse_noisy
    )


    rmse_denoised = compute_rmse(
        mse_denoised
    )


    psnr_noisy = compute_psnr(
        mse_noisy
    )


    psnr_denoised = compute_psnr(
        mse_denoised
    )


    improvement = (
        (mse_noisy - mse_denoised)
        / mse_noisy
    ) * 100


    plt.figure(
        figsize=(18, 5)
    )


    plt.subplot(1, 4, 1)

    plt.imshow(
        original_image
    )

    plt.title(
        "1. Original Image"
    )

    plt.axis("off")


    plt.subplot(1, 4, 2)

    plt.imshow(
        noisy_image
    )

    plt.title(
        "2. Adding Gaussian Noise"
    )

    plt.axis("off")


    plt.subplot(1, 4, 3)

    plt.imshow(
        encoded_image[0, :, :, 0],
        cmap="gray"
    )

    plt.title(
        "3. Encoding Step"
    )

    plt.axis("off")


    plt.subplot(1, 4, 4)

    plt.imshow(
        np.clip(
            denoised_image,
            0,
            1
        )
    )

    plt.title(
        "4. Decoded / Denoised Image"
    )

    plt.axis("off")


    plt.tight_layout()

    plt.show()


    print("\nFINAL RESULT")

    print(
        "Original Image:",
        img_size,
        "x",
        img_size,
        "x 3"
    )

    print(
        "Noise Factor:",
        noise_factor
    )

    print(
        "Encoded Representation:",
        encoded_image.shape[1],
        "x",
        encoded_image.shape[2],
        "x",
        encoded_image.shape[3]
    )


    print("\nBEFORE DENOISING")

    print(
        "MSE:",
        "{:.8f}".format(
            float(mse_noisy)
        )
    )

    print(
        "RMSE:",
        "{:.8f}".format(
            float(rmse_noisy)
        )
    )

    print(
        "PSNR:",
        "{:.2f}".format(
            float(psnr_noisy)
        ),
        "dB"
    )


    print("\nAFTER DENOISING")

    print(
        "MSE:",
        "{:.8f}".format(
            float(mse_denoised)
        )
    )

    print(
        "RMSE:",
        "{:.8f}".format(
            float(rmse_denoised)
        )
    )

    print(
        "PSNR:",
        "{:.2f}".format(
            float(psnr_denoised)
        ),
        "dB"
    )


    print(
        "\nNoise Reduction Improvement:",
        "{:.2f}".format(
            improvement
        ),
        "%"
    )


    print(
        "\nImage denoising completed successfully!"
    )


if __name__ == "__main__":
    main()